We are using https://www.litellm.ai


In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [4]:
# Load API keys from a .env file
# Create a .env file in the same folder with:
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...
# GROQ_API_KEY=gsk_...

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
Anthropic key loaded:  ❌
Groq key loaded:       ✅


The Simplest LiteLLM Example — Unified API
- The biggest pain point: every provider has a different SDK.
- LiteLLM gives you one function — completion() — that works with all of them. Look at how clean this is:

In [5]:
from litellm import completion

# Call Gemini
response_gemini = completion(
    model="gemini/gemini-3.6-flash",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in one sentence."
        }
    ]
)

print("🔵 Gemini:", response_gemini.choices[0].message.content)


# Call Groq

# Call Groq (super fast inference)
response_groq = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🟢 Groq:      ", response_groq.choices[0].message.content)

🔵 Gemini: **Retrieval-Augmented Generation (RAG)** is an AI technique that improves the accuracy and reliability of a large language model by fetching relevant facts from an external database before generating a response.
🟢 Groq:       RAG (Retrieval‑Augmented Generation) is a hybrid AI approach that first pulls relevant external documents or data, then uses those retrieved pieces to inform and enrich the language model’s generation of more accurate, context‑aware responses.


In [6]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/openai/gpt-oss-20b"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-3.6-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")
# so we dont have api key of openai and anthriopic so that will not show any result therefore model will switch to groq or gemini. 

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : Retrieval‑Augmented Generation (RAG) is a technique that improves a language mod
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : **Retrieval-Augmented Generation (RAG)** is an AI technique that enhances large 


Automatic Fallbacks — When OpenAI Goes Down
- Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.
- With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [ ]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "groq/openai/gpt-oss-20b",       # 1sty backup
        "gemini/gemini-3.6-flash"        # 2nd backup
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

20:12:38 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "d:\GenAI\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2834, in async_completion
    response: Final = await client.post(
                      ^^^^^^^^^^^^^^^^^^
  File "d:\GenAI\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 300, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\GenAI\.venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 721, in post
 

Response: ## What is an **LLM Gateway**?

An **LLM Gateway** is a software layer (sometimes called a “gateway,” “proxy,” or “router”) that sits between your application code and one or more large‑language‑model ...

Which model actually answered? openai/gpt-oss-20b


In [ ]:
# gemini/gemini-1.5-flash failed so model get fallback to openai/gpt-oss-20b and this answered the query.

Cost Tracking - Know Where Your Money Goes
- LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.

In [14]:
from litellm import completion, completion_cost

response = completion(
    model="gemini/gemini-3.6-flash",
    messages=[
        {"role": "user", "content": "Write a haiku about AI."}
    ]
)

cost = completion_cost(
    completion_response=response
)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Silent code awakes,
Thinking without a heartbeat,
Future learns to speak.

Input tokens:  8
Output tokens: 708
Cost:         $0.00266100


Caching - Don't Pay Twice for the Same Question
- If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.
- Enable in-memory caching with one line:

In [15]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [18]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="gemini/gemini-3.6-flash",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"❄️  First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

❄️  First call (API):   1.87s — LLM stands for Large Language Model.
⚡ Second call (cache): 0.4954s — LLM stands for Large Language Model.

🚀 Speedup: 3.8x faster, and ZERO cost on the second call!


Smart Routing - The Right Model for the Right Job

Why use one model for everything?
- Coding tasks → Claude Sonnet
- Cheap summaries → GPT-4o-mini
- Super fast replies → Groq Llama
- Complex reasoning → Claude Opus
- Use LiteLLM's Router to define routing rules:

In [21]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gemini/gemini-3.6-flash",
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="balanced",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 balanced (gemini):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  AI is reshaping software development in three big ways:  

1. **Automation & Productivity** – Code generation, debugging, and testing are now assisted

🧠 balanced (gemini):
 Here is the most simple and standard way to reverse a string in Python using **slicing**:

```python
def reverse_string(text: str) -> str:
    """Reverses the given string using slicing."""
    return text[::-1]


# Example usage:
original = "Hello, World!"
reversed_text = reverse_string(original)




Load Balancing Across Multiple API Keys
- Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [31]:
from litellm import Router
import os

mmodel_list = [
    {
        "model_name": "SMART",
        "litellm_params": {
            "model": "gemini/gemini-3.6-flash",
            "api_key": os.getenv("GEMINI_API_KEY"),
        },
        "model_info": {"id": "gemini"}
    },

    {
        "model_name": "SMART",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

for i in range(6):
    r = router.completion(
        model="SMART",
        messages=[
            {"role": "user", "content": f"Say hello, request {i+1}"}
        ]
    )

    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]

    print(
        f"#{i+1:<9}"
        f"{deployment_id:<15}"
        f"{latency:>6.0f} ms   "
        f"{answer}"
    )

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle",
)

print(f"{'Request':<10}{'Deployment':<15}{'Latency':<12}{'Response':<40}")
print("-" * 77)

for i in range(30):
    r = router.completion(
        model="SMART",
        messages=[
            {"role": "user", "content": f"Say hello, request {i+1}"}
        ]
    )

    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]

    print(
        f"#{i+1:<4}"
        f"{deployment_id:<15}"
        f"{latency:>6.0f} ms   "
        f"{answer}"
    )

Task was destroyed but it is pending!
task: <Task pending name='Task-295' coro=<AsyncClient.aclose() running at d:\GenAI\.venv\Lib\site-packages\httpx\_client.py:1985>>


#1        groq              626 ms   Hello!
#2        groq              785 ms   Hello! Could you let me know what y
#3        groq              878 ms   Hello!  
Please tell me what you’d 
#4        groq              732 ms   Hello! Request 4.
#5        groq             1134 ms   Hello! I request 5.
#6        groq              984 ms   Hello.  

I’m sorry, but I can’t co
Request   Deployment     Latency     Response                                
-----------------------------------------------------------------------------
#1   groq              650 ms   Hello! How can I help you with requ
#2   groq              510 ms   hello
#3   groq              831 ms   Hello!  
Could you please share **t
#4   groq              563 ms   Hello! 👋

Could you let me know wha
#5   groq              840 ms   Hello! Could you please list five i
#6   groq              830 ms   Could you clarify what you mean by 
#7   groq             1120 ms   Hello! Could you let me know what y
#8   groq              

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kcprpbz6fhh9xed1gt5aptvw` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7980, Requested 571. Please try again in 4.1325s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Received Model Group=SMART
Available Model Group Fallbacks=None